In [ ]:
#переменное окружение
conda install -c conda-forge -c bioconda \
  star bwa samtools bedtools ucsc-bedgraphtobigwig requests tqdm

In [ ]:
#скачиваем Raw
python3 scripts/download_raw_reads.py --sample MoPh11
python3 scripts/download_raw_reads.py --sample MoPh14
python3 scripts/download_raw_reads.py --sample MoPh15

In [ ]:
#индексирование референса

STAR \
  --runThreadN 32 \
  --runMode genomeGenerate \
  --genomeDir day1_HiC_practice/data/reference/star_index \
  --genomeFastaFiles day1_HiC_practice/data/reference/T2T_human.fna \
  --genomeSAindexNbases 13

In [ ]:
#папки
mkdir -p results/fastqc
mkdir -p results/star/rnaseq results/logs
mkdir -p results/tracks/rnaseq
mkdir -p results/bwa/rnaseq
mkdir -p results/star/cage results/tracks/cage

In [ ]:
#посмотреть статистику STAR
cat "results/star/rnaseq/${SAMPLE}_Log.final.out"

## Пайплайн

In [ ]:
#проверка качества raw-файлов
SAMPLE="MoPh15"

fastqc \
  #data/raw/rnaseq/${SAMPLE}_R1.fastq.gz \
  #data/raw/rnaseq/${SAMPLE}_R2.fastq.gz \
  data/raw/cage/${SAMPLE}_R1.fastq.gz \
  -o results/fastqc

multiqc results/fastqc -o results/fastqc


#Выравниваем RNA-seq
THREADS=64
STAR_INDEX="data/reference/star_index"
R1="data/raw/rnaseq/${SAMPLE}_R1.fastq.gz"
R2="data/raw/rnaseq/${SAMPLE}_R2.fastq.gz"
PREFIX="results/star/rnaseq/${SAMPLE}_"
STAR \
  --runThreadN "$THREADS" \
  --genomeDir "$STAR_INDEX" \
  --readFilesIn "$R1" "$R2" \
  --readFilesCommand zcat \
  --outSAMtype BAM SortedByCoordinate \
  --outFileNamePrefix "$PREFIX" \
  --outSAMattrRGline "ID:${SAMPLE}_rnaseq" "SM:${SAMPLE}" "PL:ILLUMINA"
mv \
  "results/star/rnaseq/${SAMPLE}_Aligned.sortedByCoord.out.bam" \
  "results/star/rnaseq/${SAMPLE}.rnaseq.STAR.bam"

#индексируем Bam, делаем один bam с хорошо выровненными ридами
samtools index -@ "$THREADS" "results/star/rnaseq/${SAMPLE}.rnaseq.STAR.bam"
samtools view \
  -@ "$THREADS" \
  -b \
  -q 30 \
  "results/star/rnaseq/${SAMPLE}.rnaseq.STAR.bam" \
  > "results/star/rnaseq/${SAMPLE}.rnaseq.STAR.q30.bam"

samtools index -@ "$THREADS" "results/star/rnaseq/${SAMPLE}.rnaseq.STAR.q30.bam"

#получаем RNA-seq bedGraph и bigWig
BAM="results/star/rnaseq/${SAMPLE}.rnaseq.STAR.bam"
CHROM_SIZES="data/reference/chrom.sizes"
bedtools genomecov \
  -ibam "$BAM" \
  -bg \
  -split \
  > "results/tracks/rnaseq/${SAMPLE}.rnaseq.STAR.bedGraph"
#сортируем bedGraph
sort -k1,1 -k2,2n \
  "results/tracks/rnaseq/${SAMPLE}.rnaseq.STAR.bedGraph" \
  > "results/tracks/rnaseq/${SAMPLE}.rnaseq.STAR.sorted.bedGraph"
#конвертируем в bigWig
bedGraphToBigWig \
  "results/tracks/rnaseq/${SAMPLE}.rnaseq.STAR.sorted.bedGraph" \
  "$CHROM_SIZES" \
  "results/tracks/rnaseq/${SAMPLE}.rnaseq.STAR.bw"

#выравниваем RNA-seq с помощью bwa
THREADS=64
BWA_INDEX="data/reference/T2T_human.fna"

R1="data/raw/rnaseq/${SAMPLE}_R1.fastq.gz"
R2="data/raw/rnaseq/${SAMPLE}_R2.fastq.gz"

bwa mem \
  -t "$THREADS" \
  "$BWA_INDEX" \
  "$R1" "$R2" \
  | samtools sort \
      -@ "$THREADS" \
      -o "results/bwa/rnaseq/${SAMPLE}.rnaseq.BWA.bam"

samtools index -@ "$THREADS" "results/bwa/rnaseq/${SAMPLE}.rnaseq.BWA.bam"


#оставим только reads с MAPQ >= 30
samtools view \
  -@ "$THREADS" \
  -b \
  -q 30 \
  "results/bwa/rnaseq/${SAMPLE}.rnaseq.BWA.bam" \
  > "results/bwa/rnaseq/${SAMPLE}.rnaseq.BWA.q30.bam"

samtools index -@ "$THREADS" "results/bwa/rnaseq/${SAMPLE}.rnaseq.BWA.q30.bam"

#делаем BWA coverage track для IGV
mkdir -p results/tracks/bwa
BAM="results/bwa/rnaseq/${SAMPLE}.rnaseq.BWA.q30.bam"
CHROM_SIZES="data/reference/chrom.sizes"
bedtools genomecov \
  -ibam "$BAM" \
  -bg \
  > "results/tracks/bwa/${SAMPLE}.rnaseq.BWA.q30.bedGraph"
sort -k1,1 -k2,2n \
  "results/tracks/bwa/${SAMPLE}.rnaseq.BWA.q30.bedGraph" \
  > "results/tracks/bwa/${SAMPLE}.rnaseq.BWA.q30.sorted.bedGraph"
bedGraphToBigWig \
  "results/tracks/bwa/${SAMPLE}.rnaseq.BWA.q30.sorted.bedGraph" \
  "$CHROM_SIZES" \
  "results/tracks/bwa/${SAMPLE}.rnaseq.BWA.q30.bw"

#выравниваем CAGE
#STAR
R1="data/raw/cage/${SAMPLE}_R1.fastq.gz"
PREFIX="results/star/cage/${SAMPLE}_"
STAR \
  --runThreadN "$THREADS" \
  --genomeDir "$STAR_INDEX" \
  --readFilesIn "$R1" \
  --readFilesCommand zcat \
  --outSAMtype BAM SortedByCoordinate \
  --outFileNamePrefix "$PREFIX" \
  --outSAMattrRGline "ID:${SAMPLE}_cage" "SM:${SAMPLE}" "PL:ILLUMINA"
#переименуем и проиндексируем BAM
mv \
  "results/star/cage/${SAMPLE}_Aligned.sortedByCoord.out.bam" \
  "results/star/cage/${SAMPLE}.cage.STAR.bam"
samtools index -@ "$THREADS" "results/star/cage/${SAMPLE}.cage.STAR.bam"
#сделаем bam с MAPQ >= 30
samtools view \
  -@ "$THREADS" \
  -b \
  -q 30 \
  "results/star/cage/${SAMPLE}.cage.STAR.bam" \
  > "results/star/cage/${SAMPLE}.cage.STAR.q30.bam"
samtools index -@ "$THREADS" "results/star/cage/${SAMPLE}.cage.STAR.q30.bam"
#делаем bedGraph и bigWig
BAM="results/star/cage/${SAMPLE}.cage.STAR.bam"
bedtools genomecov \
  -ibam "$BAM" \
  -bg \
  -split \
  > "results/tracks/cage/${SAMPLE}.cage.STAR.bedGraph"
sort -k1,1 -k2,2n \
  "results/tracks/cage/${SAMPLE}.cage.STAR.bedGraph" \
  > "results/tracks/cage/${SAMPLE}.cage.STAR.sorted.bedGraph"
bedGraphToBigWig \
  "results/tracks/cage/${SAMPLE}.cage.STAR.sorted.bedGraph" \
  "$CHROM_SIZES" \
  "results/tracks/cage/${SAMPLE}.cage.STAR.bw"